In [23]:

# 1. Setup
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score



In [5]:
# 2. Load mappings

def load_mapping(file_path):
    """Load mapping from txt file (one entity/relation per line)."""
    mapping = {}
    with open(file_path, "r") as f:
        for i, line in enumerate(f):
            mapping[line.strip()] = i
    return mapping

ent2id = load_mapping("C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/entities.txt")
rel2id = load_mapping("C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/relations.txt")

print("Num entities:", len(ent2id))
print("Num relations:", len(rel2id))

Num entities: 94046
Num relations: 107


In [6]:
val_triples = torch.load("C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/val_triples.pt")
test_triples = torch.load("C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/test_triples.pt")

print("Validation triples:", val_triples.shape)
print("Test triples:", test_triples.shape)


Validation triples: torch.Size([582709, 3])
Test triples: torch.Size([4855, 3])


In [21]:

# 4. Define ProposedRGCNModel

class ProposedRGCNModel(nn.Module):
    def __init__(self, num_entities, num_relations, dim=64, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(num_entities, dim)
        self.rel_emb = nn.Embedding(num_relations, dim)
        self.fc = nn.Linear(dim, 1)  # output 1
        self.dropout = nn.Dropout(dropout)


    def forward(self, head, rel, tail):
        h = self.emb(head)
        r = self.rel_emb(rel)
        t = self.emb(tail)
        h = self.fc(h)
        h = self.dropout(F.relu(h))
        score = torch.sum(h * r * t, dim=-1)   # DistMult-style scoring
        return score

# 5. Loader for ProposedRGCN
def load_proposed_model(model_path, num_entities, num_relations, device, dim=64):
    model = ProposedRGCNModel(num_entities, num_relations, dim=dim, dropout=0.3).to(device)
    checkpoint = torch.load(model_path, map_location=device)

    model_dict = model.state_dict()

    # Only load layers with matching shapes
    filtered_dict = {k: v for k, v in checkpoint.items() if k in model_dict and v.size() == model_dict[k].size()}
    model_dict.update(filtered_dict)
    model.load_state_dict(model_dict)

    print(" Loaded checkpoint with compatible layers only.")
    return model




In [24]:


# 6. Evaluation functions

def evaluate_proposed(model, triples, device, num_neg=10):
    """Compute AUROC, AUPRC with negative sampling."""
    model.eval()
    y_true, y_score = [], []

    with torch.no_grad():
        for h, r, t in triples:
            h = torch.tensor([h], device=device)
            r = torch.tensor([r], device=device)
            t = torch.tensor([t], device=device)

            # Positive score
            pos_score = model(h, r, t).item()
            y_true.append(1)
            y_score.append(pos_score)

            # Negative samples: corrupt tail
            for _ in range(num_neg):
                neg_t = torch.randint(0, len(ent2id), (1,), device=device)
                neg_score = model(h, r, neg_t).item()
                y_true.append(0)
                y_score.append(neg_score)

    auroc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    return {"AUROC": auroc, "AUPRC": auprc}



In [9]:
checkpoint = torch.load(prop_path, map_location=device)
print(checkpoint.keys())


odict_keys(['emb.weight', 'rel_emb.weight', 'fc.weight', 'fc.bias'])


In [25]:
# 7. Run evaluation
from torch import device


prop_path =  "C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/notebooks/checkpoints/proposed_rgcn.pt"
if os.path.exists(prop_path):
    print("\n=== Evaluating PROPOSED RGCN ===")
    proposed_model = load_proposed_model(prop_path, len(ent2id), len(rel2id), device)

    val_metrics = evaluate_proposed(proposed_model, val_triples, device, num_neg=50)
    test_metrics = evaluate_proposed(proposed_model, test_triples, device, num_neg=50)

    print("Validation metrics:", val_metrics)
    print("Test metrics:", test_metrics)

    results = {"val": val_metrics, "test": test_metrics}
    os.makedirs("results", exist_ok=True)
    with open("results/proposed_rgcn_metrics.json", "w") as f:
        json.dump(results, f, indent=2)

    print("\nSaved ProposedRGCN metrics to results/proposed_rgcn_metrics.json")
else:
    print(" ProposedRGCN checkpoint not found:", prop_path)




=== Evaluating PROPOSED RGCN ===
 Loaded checkpoint with compatible layers only.
Validation metrics: {'AUROC': 0.4994302483046772, 'AUPRC': 0.019492836724609254}
Test metrics: {'AUROC': 0.5645501759045268, 'AUPRC': 0.02194726273711984}


NameError: name 'json' is not defined